# 09 - OpenAI Agents SDK Tool Guard Pattern

This notebook shows how to put Metatate in front of tools an agent might call. The example is deterministic and does not call an LLM, so it can run offline and in CI.

In a production OpenAI Agents SDK app, the same guard function would wrap tool handlers before they execute.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

client = get_client()
mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
print(f"Metatate examples mode: {mode}")


def decision_label(response):
    data = response.get("data", {})
    decision = data.get("decision")
    if isinstance(decision, dict):
        return decision.get("decision") or decision.get("overall_decision") or data.get("overall_decision", "UNKNOWN")
    return decision or data.get("overall_decision", "UNKNOWN")


def rationale_text(response):
    data = response.get("data", {})
    decision = data.get("decision")
    if isinstance(decision, dict):
        return decision.get("rationale") or data.get("summary") or ""
    return data.get("rationale") or data.get("summary") or ""


def agent_action_text(response):
    action = response.get("data", {}).get("agent_action", {})
    if isinstance(action, dict):
        return action.get("message") or action.get("safe_next_step") or action.get("suggested_next_tool") or ""
    return ""


## Proposed tool calls


In [ ]:
proposed_tool_calls = [
    {
        "tool_call_id": "openai-001",
        "tool": "export_to_salesforce",
        "table_name": "ACMECLOUD_DEMO.PUBLIC.CUSTOMERS",
        "operation": "export",
        "intended_use": "external_sharing",
        "actor_role": "DATA_ENGINEER",
        "columns": ["CUSTOMER_ID", "CUSTOMER_NAME", "EMAIL", "ACCOUNT_STATUS"],
        "destination": {"system": "SALESFORCE", "jurisdiction": "US"},
        "consumer_jurisdiction": "EU",
    },
    {
        "tool_call_id": "openai-002",
        "tool": "export_to_ads_platform",
        "table_name": "ACMECLOUD_DEMO.PUBLIC.CUSTOMERS",
        "operation": "export",
        "intended_use": "external_sharing",
        "actor_role": "DATA_ENGINEER",
        "columns": ["CUSTOMER_ID", "CUSTOMER_NAME", "EMAIL"],
        "destination": {"system": "ADS_PLATFORM", "jurisdiction": "US"},
        "consumer_jurisdiction": "US",
    },
]


## Guard the tool call


In [ ]:
def guard_tool_call(call):
    response = client.authorize_use(
        call["table_name"],
        operation=call["operation"],
        intended_use=call["intended_use"],
        actor_role=call["actor_role"],
        columns=call.get("columns"),
        destination=call.get("destination"),
        consumer_jurisdiction=call.get("consumer_jurisdiction"),
    )
    decision = decision_label(response)
    if decision == "ALLOW":
        outcome = "execute"
    elif decision == "CONDITIONAL":
        outcome = "defer_for_controls"
    else:
        outcome = "block"
    return {
        "tool_call_id": call["tool_call_id"],
        "tool": call["tool"],
        "decision": decision,
        "outcome": outcome,
        "decision_id": response.get("data", {}).get("decision_id"),
        "message": agent_action_text(response),
    }

guard_trace = pd.DataFrame([guard_tool_call(call) for call in proposed_tool_calls])
guard_trace


In [ ]:
for record in guard_trace.to_dict(orient="records"):
    print(f"{record['tool_call_id']} -> {record['outcome']} ({record['decision']})")
    print(record["message"])
    print()


The agent can still plan creatively, but execution is constrained. Tools that move or expose governed data only run after Metatate returns an allowed or controlled path.
